In [692]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from scipy.interpolate import griddata

In [693]:
df = pd.read_csv('ToyData3.csv')
print(df.shape)
df.head()

(100, 4)


,instance_no,L2_value,drop_value,loss
0,0,0.001430,0.122449,1.016393
1,1,0.008490,0.448004,0.900728
2,2,0.007661,0.328000,0.814525
3,3,0.002625,0.179936,0.730412
4,4,0.005005,0.301888,0.679886


In [694]:
# Flipping the loss and normalizing the y axis to get AUC
df['auc'] = (-df['loss'] + df['loss'].max()) * (1 / (df['loss'].max() * df.shape[0]))
print(df.shape)
df.head()

(100, 5)


,instance_no,L2_value,drop_value,loss,auc
0,0,0.001430,0.122449,1.016393,0.000000
1,1,0.008490,0.448004,0.900728,0.001138
2,2,0.007661,0.328000,0.814525,0.001986
3,3,0.002625,0.179936,0.730412,0.002814
4,4,0.005005,0.301888,0.679886,0.003311


In [695]:
def plot_3d_scatter_surface(df, x_variable, y_variables, output_variable, surface=True, highlight=False):
    name = ['L2', 'Dropout']
    
    for i, y_var in enumerate(y_variables):
        # Find the index of the lowest output variable value if highlight is True
        lowest_index = df[output_variable].idxmin() if highlight else None
        
        # Create a scatter plot
        scatter = go.Scatter3d(
            x=df[x_variable],
            y=df[y_var],
            z=df[output_variable],
            mode='markers',
            marker=dict(
                size=[15 if i == lowest_index else 5 for i in range(len(df))],  # Highlighted point is bigger,
                color=['red' if i == lowest_index else 'grey' for i in range(len(df))] if highlight else 'grey',  # Highlight the lowest value if highlight is True
                colorscale='Viridis',  # Choose a color scale
                opacity=0.8,
            ),
            name=output_variable
        )

        # Create surface data
        x_data = df[x_variable]
        y_data = df[y_var]
        z_data = df[output_variable]

        x_range = np.linspace(min(x_data), max(x_data), 100)
        y_range = np.linspace(min(y_data), max(y_data), 100)
        X, Y = np.meshgrid(x_range, y_range)
        Z = griddata((x_data, y_data), z_data, (X, Y), method='cubic')

        if surface:
            # Create a 3D surface
            surface = go.Surface(
                x=X,
                y=Y,
                z=Z,
                colorscale='Jet',  # Choose a color scale
                opacity=0.6
            )

            # Create the figure
            fig = go.Figure(data=[scatter, surface])

        else:
            fig = go.Figure(data=[scatter])

        # Update layout
        fig.update_layout(
            scene=dict(
                xaxis_title=x_variable,
                yaxis_title=y_var,
                zaxis_title=output_variable
            ),
            width=800,  # adjust width
            height=600,  # adjust height
            title=f'Mean Loss for different values of {name[i]} regularization'
        )

        # Show the plot
        fig.show()

In [696]:
plot_3d_scatter_surface(df, 'instance_no', ['L2_value', 'drop_value'], 'loss', surface=True, highlight=True)

In [697]:
def plot_2d_scatter(df, x_column, output_column, highlight=False):

    # Find the index of the lowest output variable value if highlight is True
    lowest_index = df[output_column].idxmin() if highlight else None
    
    # Create a scatter plot
    scatter = go.Scatter(
        x=df[x_column],
        y=df[output_column],
        mode='lines+markers',
    
        marker=dict(
            size=[15 if i == lowest_index else 5 for i in range(len(df))],  # Highlighted point is bigger
            color=['red' if i == lowest_index else 'blue' for i in range(len(df))] if highlight else 'blue',  # Highlight the lowest value if highlight is True
            opacity=0.8,
        ),
        name=output_column
    )

    # Create the figure
    fig = go.Figure(data=[scatter])

    # Update layout
    fig.update_layout(
        xaxis_title=x_column,
        yaxis_title=output_column,
        # yaxis=dict(tickformat='e'),
        width=800,  # adjust width
        height=600,  # adjust height
        title=f'Loss as a function of iterations'
    )

    # Show the plot
    fig.show()

In [698]:
plot_2d_scatter(df, 'instance_no', 'loss', highlight=True)

In [699]:
auc = np.trapz(df['auc'], x=df['instance_no'])
print(f'Area under the curve (AUC): {auc:.2f}')

plot_2d_scatter(df, 'instance_no', 'auc', highlight=False)

Area under the curve (AUC): 0.89
